# Modelado — comparativa, optimización y modelo final

Fase 5-6 de la guía (pasos 25-31) + persistencia (paso 36). Consume el pipeline
cerrado por las fases anteriores:

- `data/processed/train_features.csv` — dataset con las variables del EDA/FE
  (`feature_engineering.ipynb`).
- `src/preprocessing.build_features()` — filtra días de cierre, aplica one-hot a
  `grupo_dia`/`temporada`, recalcula la tendencia y elimina columnas redundantes
  (`feature_preprocessing.ipynb`).
- `src/models/scaler.joblib` — StandardScaler de `dias_desde_inicio`, ajustado
  sobre train por Preprocesado.

**El conjunto de test no se toca en este notebook.** Toda la selección de modelo
e hiperparámetros se hace con validación cruzada temporal sobre train; la
evaluación única contra test vive en `notebooks/evaluation.ipynb`.

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np
import pandas as pd
import joblib

from src.utils.preprocessing import build_features
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## 1. Carga y preprocesado

Cargamos el dataset de features y aplicamos exactamente el mismo preprocesado
que definió el lane de Preprocesado. Guardamos aparte las fechas (que
`build_features` descarta) porque las necesitamos para los folds temporales y
para el baseline estacional.

In [2]:
df = pd.read_csv('../data/processed/train_features.csv', parse_dates=['fecha_cita'])
df = df.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)

# mismo filtro de cierres que build_features, aplicado antes para conservar la fecha alineada
df_model = df[df['es_cierre'] != 1].reset_index(drop=True)

X, y = build_features(df_model)
fechas = df_model['fecha_cita']
tramos = df_model['tramo']

# escalado de la tendencia con el scaler ajustado por Preprocesado
scaler = joblib.load('../src/models/scaler.joblib')
X[['dias_desde_inicio']] = scaler.transform(X[['dias_desde_inicio']])

print(f"Filas: {X.shape[0]} (de {len(df)} tras excluir {len(df)-len(df_model)} tramos de cierre)")
print(f"Features ({X.shape[1]}): {list(X.columns)}")
print(f"Rango temporal: {fechas.min().date()} -> {fechas.max().date()}")

Filas: 1240 (de 1252 tras excluir 12 tramos de cierre)
Features (15): ['dia_semana', 'mes', 'trimestre', 'dias_desde_inicio', 'tramo_tarde', 'es_festivo', 'es_vispera_festivo', 'es_fecha_comercial', 'grupo_dia_entre_semana', 'grupo_dia_fin_de_semana', 'grupo_dia_viernes', 'temporada_invierno', 'temporada_otoño', 'temporada_primavera', 'temporada_verano']
Rango temporal: 2024-05-09 -> 2026-01-24


## 2. Métrica de evaluación (paso 25)

- **MAE (principal):** se lee directamente como "citas de más o de menos por
  tramo", la unidad con la que el negocio decide personal en sala.
- **RMSE (secundaria):** penaliza los errores grandes — infra-dotar un sábado
  por la tarde cuesta más que varios fallos pequeños.
- **R² (referencia):** cuánta varianza explica el modelo; sin unidades de negocio.
- **MAPE descartado:** el 21% de los tramos de mañana tienen 0 citas (EDA §2) y
  el error porcentual se dispara a infinito.

In [3]:
def evaluar(y_true, y_pred, nombre=''):
    return {
        'modelo': nombre,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

tscv = TimeSeriesSplit(n_splits=5)
folds = list(tscv.split(X))
for i, (tr, va) in enumerate(folds, 1):
    print(f"Fold {i}: train {len(tr):4d} filas (hasta {fechas.iloc[tr].max().date()}) "
          f"-> valida {len(va):3d} filas ({fechas.iloc[va].min().date()} a {fechas.iloc[va].max().date()})")

Fold 1: train  210 filas (hasta 2024-08-21) -> valida 206 filas (2024-08-22 a 2024-12-02)
Fold 2: train  416 filas (hasta 2024-12-02) -> valida 206 filas (2024-12-03 a 2025-03-18)
Fold 3: train  622 filas (hasta 2025-03-18) -> valida 206 filas (2025-03-19 a 2025-06-29)
Fold 4: train  828 filas (hasta 2025-06-29) -> valida 206 filas (2025-06-30 a 2025-10-10)
Fold 5: train 1034 filas (hasta 2025-10-10) -> valida 206 filas (2025-10-11 a 2026-01-24)


## 3. Baselines por validación cruzada (paso 26)

Dos referencias mínimas, evaluadas **en los mismos folds temporales** que los
modelos para que la comparación sea justa:

1. **Media del fold de train**: ignora toda estructura.
2. **Estacional ingenuo (t-7)**: el valor real del mismo tramo 7 días naturales
   antes (vía join por fecha, robusto a huecos por cierres). Solo usa el pasado,
   así que no hay fuga temporal.

In [4]:
# predicción t-7 por join fecha-7 días dentro del mismo tramo
hist = df_model[['fecha_cita', 'tramo', 'n_citas']].copy()
hist_shift = hist.rename(columns={'n_citas': 'pred_t7'})
hist_shift['fecha_cita'] = hist_shift['fecha_cita'] + pd.Timedelta(days=7)
t7 = hist.merge(hist_shift, on=['fecha_cita', 'tramo'], how='left')['pred_t7']

filas_cv = []
for tr, va in folds:
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    filas_cv.append(evaluar(y_va, np.full(len(va), y_tr.mean()), 'Baseline media'))
    mask = t7.iloc[va].notna()
    filas_cv.append(evaluar(y_va[mask.values], t7.iloc[va][mask.values], 'Baseline estacional t-7'))

base_cv = pd.DataFrame(filas_cv).groupby('modelo').mean().round(2)
print(f"Cobertura t-7 en validación: {t7.notna().mean():.0%} de las filas")
base_cv

Cobertura t-7 en validación: 98% de las filas


,MAE,RMSE,R2
modelo,,,
Baseline estacional t-7,2.02,2.64,0.07
Baseline media,2.30,3.02,-0.21


## 4. Comparativa de modelos (paso 27)

Seis algoritmos con parámetros por defecto, mismos folds. Lineales como
referencia interpretable; árboles y ensembles porque el EDA anticipó
interacciones no lineales (el efecto fin de semana se dispara en temporada
alta); KNN como no-paramétrico basado en distancias.

In [5]:
modelos = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'KNN': KNeighborsRegressor(),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

filas = []
for nombre, modelo in modelos.items():
    for tr, va in folds:
        m = modelo.__class__(**modelo.get_params())
        m.fit(X.iloc[tr], y.iloc[tr])
        filas.append(evaluar(y.iloc[va], m.predict(X.iloc[va]), nombre))

comparativa = pd.DataFrame(filas).groupby('modelo').mean().sort_values('MAE').round(2)
comparativa

,MAE,RMSE,R2
modelo,,,
GradientBoosting,1.88,2.41,0.20
RandomForest,1.89,2.43,0.17
LinearRegression,1.92,2.37,0.24
Ridge,1.94,2.43,0.22
DecisionTree,2.15,2.81,-0.13
KNN,2.24,2.90,-0.16


**Lectura:** la tabla anterior (media de los 5 folds temporales) decide qué
modelos pasan a optimización. Cualquier candidato debe batir a los dos
baselines — si no lo hace, no está aprendiendo nada que el calendario simple
no sepa ya.

## 5. Optimización de hiperparámetros (pasos 28-31)

Optimizamos los **dos mejores** de la comparativa (esperablemente los
ensembles). `RandomizedSearchCV` en lugar de búsqueda exhaustiva porque el
espacio combinado es grande, siempre con los mismos folds temporales
(`TimeSeriesSplit`) y optimizando MAE.

In [6]:
top2 = comparativa.index[:2].tolist()
print('Modelos a optimizar:', top2)

espacios = {
    'RandomForest': (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {
            'n_estimators': [200, 400, 600],
            'max_depth': [None, 6, 10, 16],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 0.5, 1.0],
        },
    ),
    'GradientBoosting': (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {
            'n_estimators': [150, 300, 500],
            'learning_rate': [0.03, 0.05, 0.1],
            'max_depth': [2, 3, 4],
            'min_samples_leaf': [1, 5, 10],
            'subsample': [0.8, 1.0],
        },
    ),
    'Ridge': (
        Ridge(random_state=RANDOM_STATE),
        {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    ),
    'KNN': (
        KNeighborsRegressor(),
        {'n_neighbors': [3, 5, 9, 15, 25], 'weights': ['uniform', 'distance']},
    ),
    'DecisionTree': (
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {'max_depth': [3, 5, 8, 12, None], 'min_samples_leaf': [1, 5, 10, 20]},
    ),
    'LinearRegression': (LinearRegression(), {}),
}

busquedas = {}
for nombre in top2:
    est, grid = espacios[nombre]
    if not grid:
        continue
    rs = RandomizedSearchCV(
        est, grid, n_iter=25, cv=tscv, scoring='neg_mean_absolute_error',
        random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    rs.fit(X, y)
    busquedas[nombre] = rs
    print(f"\n{nombre}: mejor MAE CV = {-rs.best_score_:.3f}")
    print(f"  mejores parámetros: {rs.best_params_}")

Modelos a optimizar: ['GradientBoosting', 'RandomForest']



GradientBoosting: mejor MAE CV = 1.744
  mejores parámetros: {'subsample': 0.8, 'n_estimators': 150, 'min_samples_leaf': 1, 'max_depth': 3, 'learning_rate': 0.03}



RandomForest: mejor MAE CV = 1.744
  mejores parámetros: {'n_estimators': 200, 'min_samples_leaf': 8, 'max_features': 1.0, 'max_depth': 6}


In [7]:
resumen = []
for nombre in top2:
    if nombre in busquedas:
        resumen.append({
            'modelo': nombre,
            'MAE CV (defecto)': comparativa.loc[nombre, 'MAE'],
            'MAE CV (optimizado)': round(-busquedas[nombre].best_score_, 2),
        })
tabla_opt = pd.DataFrame(resumen).set_index('modelo')
tabla_opt['mejora'] = (tabla_opt['MAE CV (defecto)'] - tabla_opt['MAE CV (optimizado)']).round(2)
tabla_opt

,MAE CV (defecto),MAE CV (optimizado),mejora
modelo,,,
GradientBoosting,1.88,1.74,0.14
RandomForest,1.89,1.74,0.15


**Lectura:** documentamos el impacto de la optimización frente a los
parámetros por defecto (paso 31). El ganador se elige por MAE de CV — no por
test, que sigue intacto.

## 6. Modelo final y persistencia (paso 36)

`RandomizedSearchCV` con `refit=True` ya reentrena el mejor estimador sobre
**todo el train**. Guardamos un artefacto autocontenido con lo necesario para
inferencia y para la evaluación final:

- el modelo entrenado,
- la lista de columnas de X (para alinear el one-hot de test),
- metadatos de trazabilidad (métrica CV, fecha, features de origen).

El scaler de `dias_desde_inicio` ya está guardado por Preprocesado en
`src/models/scaler.joblib` — no lo duplicamos.

In [8]:
ganador = min(busquedas, key=lambda n: -busquedas[n].best_score_)
modelo_final = busquedas[ganador].best_estimator_

artefacto = {
    'modelo': modelo_final,
    'nombre': ganador,
    'columnas': list(X.columns),
    'mae_cv': round(-busquedas[ganador].best_score_, 3),
    'entrenado_hasta': str(fechas.max().date()),
    'mejores_params': busquedas[ganador].best_params_,
}

os.makedirs('../src/models', exist_ok=True)
ruta = '../src/models/modelo_ocupacion.joblib'
joblib.dump(artefacto, ruta)
print(f"Guardado: {ruta}")
print(f"Modelo: {ganador} | MAE CV: {artefacto['mae_cv']} | entrenado hasta {artefacto['entrenado_hasta']}")
print("\nCómo cargarlo:")
print("  art = joblib.load('src/models/modelo_ocupacion.joblib')")
print("  modelo, columnas = art['modelo'], art['columnas']")

Guardado: ../src/models/modelo_ocupacion.joblib
Modelo: RandomForest | MAE CV: 1.744 | entrenado hasta 2026-01-24

Cómo cargarlo:
  art = joblib.load('src/models/modelo_ocupacion.joblib')
  modelo, columnas = art['modelo'], art['columnas']


## 7. Qué queda para `evaluation.ipynb`

- Evaluación **única** contra test con este artefacto (paso 32).
- Análisis de residuos, real vs. predicho en el tiempo y por tramo (paso 33).
- Interpretabilidad: importancia de features del modelo final (paso 34).
- Contraste con el problema de negocio y limitaciones (paso 35).